# Notações Arquiteturais na Prática: Architecture-as-Code (C4, Componentes e 4+1)

Neste notebook executável, exploramos como representar modelos arquiteturais de forma estruturada em Python (**Architecture-as-Code**) para:
1. Modelar a hierarquia do **Modelo C4** (Contexto, Contêineres e Componentes) e contratos de **Diagramas de Componentes UML** (interfaces fornecidas e requeridas).
2. Gerar automaticamente diagramas visuais na sintaxe **Mermaid** a partir do modelo em memória, evitando a dessincronização entre código e documentação (*diagram drift*).
3. Executar **funções de aptidão arquitetural (*Architectural Fitness Functions*)** para detectar violações de fronteira, dependências cíclicas e contratos não satisfeitos.
4. Mapear a rastreabilidade entre os elementos técnicos e as visões do **Modelo 4+1**.

In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict
from typing import List, Dict, Set, Tuple

@dataclass
class ComponenteUML:
    nome: str
    conteiner: str
    responsabilidade: str
    interfaces_fornecidas: List[str] = field(default_factory=list)  # Lollipop
    interfaces_requeridas: List[str] = field(default_factory=list)  # Socket

@dataclass
class ConteinerC4:
    id: str
    nome: str
    tipo: str  # ex: 'WebApp', 'API', 'Database', 'ExternalSystem'
    tecnologia: str
    responsabilidade: str

@dataclass
class RelacionamentoC4:
    origem: str
    destino: str
    descricao: str
    protocolo: str

print("Estruturas base de modelagem C4 e UML carregadas com sucesso.")

## 1. Definindo a Arquitetura de um Sistema Hospitalar (Nível 2 C4 e Componentes UML)

Vamos modelar a **Plataforma de Agendamento e Regulação de Plantões**, definindo seus contêineres C4, seus fluxos de comunicação e os componentes internos com interfaces fornecidas (*Provided*) e requeridas (*Required*).

In [ ]:
conteineres = [
    ConteinerC4("PortalWeb", "Portal de Escalas", "WebApp", "React / TypeScript", "Interface de gestão de plantões"),
    ConteinerC4("ApiPlantoes", "API de Regulação", "API", "Python / FastAPI", "Orquestra regras trabalhistas e alocações"),
    ConteinerC4("BancoClinico", "Banco de Dados Clínico", "Database", "PostgreSQL 16", "Armazena escalas, médicos e auditoria"),
    ConteinerC4("GatewayNotif", "Serviço de Mensageria", "ExternalSystem", "API HTTPS / SMS", "Dispara alertas de troca de plantão")
]

relacionamentos = [
    RelacionamentoC4("PortalWeb", "ApiPlantoes", "Solicita alocação de turno", "JSON / HTTPS"),
    RelacionamentoC4("ApiPlantoes", "BancoClinico", "Persiste escalas com lock transacional", "SQL / TCP"),
    RelacionamentoC4("ApiPlantoes", "GatewayNotif", "Notifica médico escalado", "Webhook / HTTPS")
]

componentes = [
    ComponenteUML(
        nome="MotorAlocacaoPlantao",
        conteiner="ApiPlantoes",
        responsabilidade="Valida conflitos de horário e aloca especialistas",
        interfaces_fornecidas=["IAlocacaoService"],
        interfaces_requeridas=["IRegraDescansoValidator", "IPersistenciaEscala", "INotificadorPlantao"]
    ),
    ComponenteUML(
        nome="ValidadorTrabalhista",
        conteiner="ApiPlantoes",
        responsabilidade="Verifica intervalo mínimo interjornada",
        interfaces_fornecidas=["IRegraDescansoValidator"],
        interfaces_requeridas=[]
    ),
    ComponenteUML(
        nome="RepositorioEscalas",
        conteiner="ApiPlantoes",
        responsabilidade="Acesso transacional ao PostgreSQL",
        interfaces_fornecidas=["IPersistenciaEscala"],
        interfaces_requeridas=[]
    ),
    ComponenteUML(
        nome="AdaptadorNotificacao",
        conteiner="ApiPlantoes",
        responsabilidade="Integra com o gateway externo de mensagens",
        interfaces_fornecidas=["INotificadorPlantao"],
        interfaces_requeridas=[]
    )
]

print(f"Modelo definido: {len(conteineres)} contêineres C4 e {len(componentes)} componentes UML.")

## 2. Geração Automática de Diagramas Mermaid (C4 Contêineres e Componentes UML)

A função abaixo transforma o modelo estruturado diretamente em código **Mermaid** pronto para ser renderizado em documentação Quarto ou GitHub.

In [ ]:
def gerar_mermaid_c4(conteineres: List[ConteinerC4], rels: List[RelacionamentoC4]) -> str:
    linhas = ["flowchart LR"]
    for c in conteineres:
        if c.tipo == "Database":
            linhas.append(f'    {c.id}[("{c.nome}\n[{c.tecnologia}]\n{c.responsabilidade}")]')
        else:
            linhas.append(f'    {c.id}["{c.nome}\n[{c.tecnologia}]\n{c.responsabilidade}"]')
    linhas.append("")
    for r in rels:
        linhas.append(f'    {r.origem} -->|"{r.descricao}\n[{r.protocolo}]"| {r.destino}')
    return "\n".join(linhas)

print("=== DIAGRAMA MERMAID GERADO (C4 NÍVEL 2) ===\n")
print(gerar_mermaid_c4(conteineres, relacionamentos))

## 3. Validador de Contratos UML, Fronteiras C4 e Rastreabilidade 4+1

Para garantir que a arquitetura modelada não viole princípios de engenharia de software, aplicamos três verificações automáticas:
- **Verificação 1 (Isolamento de Dados C4):** Nenhum contêiner de apresentação (`WebApp`) pode acessar diretamente um `Database`.
- **Verificação 2 (Satisfação de Interfaces UML):** Toda interface requerida (*socket*) deve corresponder a uma interface fornecida (*lollipop*) por algum componente.
- **Verificação 3 (Matriz 4+1):** Mapeamento de cobertura entre as quatro visões técnicas.

In [ ]:
def validar_arquitetura(conteineres: List[ConteinerC4], rels: List[RelacionamentoC4], comps: List[ComponenteUML]):
    tipos = {c.id: c.tipo for c in conteineres}
    erros = []

    # 1. Regra de fronteira C4: WebApp não acessa Database diretamente
    for r in rels:
        if tipos.get(r.origem) == "WebApp" and tipos.get(r.destino) == "Database":
            erros.append(f"VIOLAÇÃO C4: {r.origem} acessa diretamente o banco {r.destino} sem API intermediária.")

    # 2. Regra de contratos UML: toda interface requerida deve ser fornecida
    fornecidas = {iface for c in comps for iface in c.interfaces_fornecidas}
    for c in comps:
        for req in c.interfaces_requeridas:
            if req not in fornecidas:
                erros.append(f"VIOLAÇÃO UML: Componente '{c.nome}' requer '{req}', mas nenhum componente a fornece.")

    if not erros:
        print("[APROVADO] Todas as regras de fronteira C4 e contratos de interface UML foram satisfeitas!")
    else:
        for e in erros:
            print(f"[ERRO] {e}")

validar_arquitetura(conteineres, relacionamentos, componentes)

# Teste negativo: simulando uma violação de acesso direto ao banco e interface ausente
print("\n--- Simulando cenário com violações arquiteturais ---")
rels_invalidos = relacionamentos + [RelacionamentoC4("PortalWeb", "BancoClinico", "Query direta", "SQL")]
comps_invalidos = componentes + [ComponenteUML("RelatorioFinanceiro", "ApiPlantoes", "Gera PDF", [], ["IServicoFaturamentoInexistente"])]
validar_arquitetura(conteineres, rels_invalidos, comps_invalidos)

## Conclusão

A combinação entre **C4**, **Diagramas de Componentes UML** e o **Modelo 4+1** permite alinhar comunicação executiva, modularidade de código e requisitos operacionais. Quando representados como código (*Architecture-as-Code*), esses modelos deixam de ser desenhos estáticos e passam a atuar como especificações verificáveis ao longo de todo o ciclo de vida do sistema.